# Read, Display, and Quantify Uncertainty in NetCDF (`.nc`) Data

This notebook is designed for the **Uncertainty-aware Copernicus Indicators Dashboard**.

It shows how to:

1. Read NetCDF data using `xarray`
2. Inspect variables and coordinates
3. Select a variable such as UTCI, temperature, or an energy indicator
4. Plot maps and time series
5. Compute ensemble uncertainty statistics
6. Estimate probability of exceedance
7. Classify confidence from uncertainty spread

It works best with Copernicus `.nc` files such as ERA5-HEAT UTCI, ERA5, or projection datasets.

## 1. Install required packages

Run this once if packages are missing.

In [1]:
# Uncomment if needed
# !pip install xarray netCDF4 h5netcdf matplotlib pandas numpy

## 2. Import libraries

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

ModuleNotFoundError: No module named 'xarray'

In [3]:
!pip install xarray

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Set your NetCDF file path

Change `DATA_FILE` to your downloaded file.

In [ ]:
# Example paths:
# DATA_FILE = "data/c3s_era5_heat_utci.nc"
# DATA_FILE = "data/sis_energy_projection.nc"

DATA_FILE = "c3s_era5_heat_utci.nc"

path = Path(DATA_FILE)
if not path.exists():
    print(f"File not found: {path}")
    print("Update DATA_FILE to point to your NetCDF file.")
else:
    print(f"Using file: {path.resolve()}")

## 4. Open and inspect the dataset

In [ ]:
ds = xr.open_dataset(DATA_FILE)
ds

In [ ]:
print("Data variables:")
for v in ds.data_vars:
    print(" -", v, ds[v].dims, ds[v].shape)

print("
Coordinates:")
for c in ds.coords:
    print(" -", c, ds[c].dims, ds[c].shape)

## 5. Choose a variable automatically or manually

The notebook tries to find common Copernicus variable names. You can override `VAR_NAME` manually.

In [ ]:
preferred_names = [
    "utci", "universal_thermal_climate_index", "t2m", "temperature",
    "air_temperature", "tas", "electricity_demand", "demand"
]

available = list(ds.data_vars)
VAR_NAME = None

for name in preferred_names:
    if name in available:
        VAR_NAME = name
        break

if VAR_NAME is None:
    VAR_NAME = available[0]

print("Selected variable:", VAR_NAME)
da = ds[VAR_NAME]
da

## 6. Convert Kelvin to Celsius if needed

Many Copernicus temperature-like variables are stored in Kelvin. This cell checks the `units` attribute.

In [ ]:
units = str(da.attrs.get("units", "")).lower()
print("Units:", units)

if units in ["k", "kelvin"]:
    da_work = da - 273.15
    da_work.attrs["units"] = "degC"
    print("Converted from Kelvin to Celsius.")
else:
    da_work = da
    print("No unit conversion applied.")

da_work

## 7. Detect coordinate and dimension names

In [ ]:
def find_first(names, candidates):
    for c in candidates:
        if c in names:
            return c
    return None

all_names = list(da_work.dims) + list(da_work.coords)

time_dim = find_first(all_names, ["time", "valid_time", "forecast_reference_time"])
lat_dim = find_first(all_names, ["latitude", "lat", "y"])
lon_dim = find_first(all_names, ["longitude", "lon", "x"])
ensemble_dim = find_first(list(da_work.dims), ["number", "realization", "ensemble", "member", "model", "gcm", "rcm"])

print("time_dim:", time_dim)
print("lat_dim:", lat_dim)
print("lon_dim:", lon_dim)
print("ensemble_dim:", ensemble_dim)
print("all dims:", da_work.dims)

## 8. Plot a spatial map

This plots the first timestep and first ensemble member/model if such dimensions exist.

In [ ]:
plot_da = da_work

if time_dim in plot_da.dims:
    plot_da = plot_da.isel({time_dim: 0})

if ensemble_dim in plot_da.dims:
    plot_da = plot_da.isel({ensemble_dim: 0})

# Drop remaining non-spatial dimensions by selecting first index
for dim in list(plot_da.dims):
    if dim not in [lat_dim, lon_dim]:
        plot_da = plot_da.isel({dim: 0})

plot_da.plot()
plt.title(f"{VAR_NAME}: first available map")
plt.show()

## 9. Select a point location

Change the latitude and longitude to your location of interest.

In [ ]:
# Example: Oslo
TARGET_LAT = 59.9
TARGET_LON = 10.7

point = da_work

if lat_dim and lon_dim and lat_dim in da_work.coords and lon_dim in da_work.coords:
    point = da_work.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest")
    print("Selected nearest point:")
    print(lat_dim, float(point[lat_dim].values))
    print(lon_dim, float(point[lon_dim].values))
else:
    print("Latitude/longitude coordinates not detected. Using full data array.")

point

## 10. Plot time series at selected point

In [ ]:
ts = point

if ensemble_dim in ts.dims:
    ts_mean = ts.mean(dim=ensemble_dim)
else:
    ts_mean = ts

for dim in list(ts_mean.dims):
    if dim != time_dim:
        ts_mean = ts_mean.isel({dim: 0})

if time_dim in ts_mean.dims:
    ts_mean.plot(marker="o")
    plt.title(f"{VAR_NAME} time series at selected point")
    plt.ylabel(da_work.attrs.get("units", VAR_NAME))
    plt.show()
else:
    print("No time dimension detected for time series plot.")
    print(ts_mean.values)

## 11. Ensemble uncertainty calculations

If an ensemble/model/member dimension exists, the notebook computes uncertainty across that dimension.

If no ensemble dimension exists, it reports that proper ensemble uncertainty is unavailable.

In [ ]:
if ensemble_dim is not None and ensemble_dim in da_work.dims:
    uncertainty_dim = ensemble_dim
    print(f"Using ensemble dimension for uncertainty: {uncertainty_dim}")
else:
    uncertainty_dim = None
    print("No ensemble dimension detected.")
    print("For proper ensemble uncertainty, use a dataset with members/models/realizations.")

In [ ]:
if uncertainty_dim:
    ens_mean = da_work.mean(dim=uncertainty_dim)
    ens_median = da_work.median(dim=uncertainty_dim)
    ens_spread = da_work.std(dim=uncertainty_dim)
    ens_p05 = da_work.quantile(0.05, dim=uncertainty_dim)
    ens_p25 = da_work.quantile(0.25, dim=uncertainty_dim)
    ens_p75 = da_work.quantile(0.75, dim=uncertainty_dim)
    ens_p95 = da_work.quantile(0.95, dim=uncertainty_dim)
else:
    ens_mean = da_work
    ens_median = None
    ens_spread = None
    ens_p05 = None
    ens_p25 = None
    ens_p75 = None
    ens_p95 = None

print("Uncertainty statistics created.")

## 12. Probability of exceedance

Example thresholds:

- UTCI heat stress: 32 °C
- PM2.5: 15 or 25 µg/m³ depending on your chosen standard
- Temperature: project-specific threshold

In [ ]:
THRESHOLD = 32.0

if uncertainty_dim:
    prob_exceed = (da_work > THRESHOLD).mean(dim=uncertainty_dim) * 100
    print(f"Probability of exceedance calculated for threshold: {THRESHOLD}")
else:
    prob_exceed = xr.where(da_work > THRESHOLD, 100.0, 0.0)
    print("No ensemble dimension: exceedance is deterministic, shown as 0% or 100%.")

prob_exceed

## 13. Confidence classification

Confidence is derived from ensemble spread:

- Low spread → High confidence
- Medium spread → Medium confidence
- High spread → Low confidence

Adjust thresholds depending on the indicator.

In [ ]:
HIGH_CONF_SPREAD = 2.0
MED_CONF_SPREAD = 5.0

if ens_spread is not None:
    confidence = xr.where(
        ens_spread < HIGH_CONF_SPREAD,
        "High",
        xr.where(ens_spread < MED_CONF_SPREAD, "Medium", "Low")
    )
else:
    confidence = None

print("Confidence classification complete.")
confidence

## 14. Fan chart / uncertainty band for selected point

In [ ]:
if uncertainty_dim and time_dim:
    mean_p = ens_mean.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest") if lat_dim and lon_dim else ens_mean
    p05_p = ens_p05.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest") if lat_dim and lon_dim else ens_p05
    p95_p = ens_p95.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest") if lat_dim and lon_dim else ens_p95

    cleaned = []
    for obj in [mean_p, p05_p, p95_p]:
        for dim in list(obj.dims):
            if dim != time_dim:
                obj = obj.isel({dim: 0})
        cleaned.append(obj)
    mean_p, p05_p, p95_p = cleaned

    plt.plot(mean_p[time_dim], mean_p, label="Ensemble mean")
    plt.fill_between(mean_p[time_dim].values, p05_p.values, p95_p.values, alpha=0.3, label="5-95% range")
    plt.axhline(THRESHOLD, linestyle="--", label=f"Threshold: {THRESHOLD}")
    plt.title(f"Uncertainty band for {VAR_NAME}")
    plt.ylabel(da_work.attrs.get("units", VAR_NAME))
    plt.legend()
    plt.show()
else:
    print("Fan chart requires both time and ensemble dimensions.")

## 15. Uncertainty map

In [ ]:
if ens_spread is not None:
    map_spread = ens_spread
    if time_dim in map_spread.dims:
        map_spread = map_spread.isel({time_dim: 0})
    for dim in list(map_spread.dims):
        if dim not in [lat_dim, lon_dim]:
            map_spread = map_spread.isel({dim: 0})
    map_spread.plot()
    plt.title(f"Uncertainty spread map for {VAR_NAME}")
    plt.show()
else:
    print("No ensemble spread available to map.")

## 16. Dashboard-ready summary table

In [ ]:
summary = {}

if lat_dim and lon_dim:
    base_point = da_work.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest")
else:
    base_point = da_work

summary["variable"] = VAR_NAME
summary["threshold"] = THRESHOLD
summary["mean_value"] = float(base_point.mean().values)
summary["max_value"] = float(base_point.max().values)

if uncertainty_dim:
    spread_point = ens_spread.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest") if lat_dim and lon_dim else ens_spread
    prob_point = prob_exceed.sel({lat_dim: TARGET_LAT, lon_dim: TARGET_LON}, method="nearest") if lat_dim and lon_dim else prob_exceed
    summary["mean_uncertainty_spread"] = float(spread_point.mean().values)
    summary["mean_probability_exceedance_percent"] = float(prob_point.mean().values)
    avg_spread = summary["mean_uncertainty_spread"]
    if avg_spread < HIGH_CONF_SPREAD:
        summary["confidence"] = "High"
    elif avg_spread < MED_CONF_SPREAD:
        summary["confidence"] = "Medium"
    else:
        summary["confidence"] = "Low"
else:
    summary["mean_uncertainty_spread"] = np.nan
    summary["mean_probability_exceedance_percent"] = float((base_point > THRESHOLD).mean().values * 100)
    summary["confidence"] = "Not available without ensemble members"

summary_df = pd.DataFrame([summary])
summary_df

## 17. Save summary output

In [ ]:
OUTPUT_CSV = "dashboard_uncertainty_summary.csv"
summary_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved summary to {OUTPUT_CSV}")

## Interpretation for the dashboard

- **Risk**: how severe the value is compared with a threshold
- **Spread**: how much ensemble members disagree
- **Confidence**: high when spread is low, low when spread is high
- **Probability of exceedance**: percentage of ensemble members above an action threshold
- **Fan chart**: shows uncertainty over time
- **Map**: shows where uncertainty is spatially concentrated

Recommended dashboard message format:

> Risk is high, but confidence is low because ensemble spread is large.

or

> Risk is moderate with high confidence because ensemble members agree.